## Просто XGBoost

In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import OrdinalEncoder

data = pd.DataFrame({
    'age': [25, 30, 35, 40, 45, 50],
    'city': ['Moscow', 'SPb', 'Moscow', 'Kazan', 'SPb', 'Moscow'],  # Категориальный
    'income': [50000, 60000, 55000, 70000, 65000, 80000],
    'education': ['high', 'medium', 'high', 'low', 'medium', 'high'],  # Категориальный
    'target': [1, 0, 1, 0, 1, 0]
})

X = data.drop('target', axis=1)
y = data['target']

cat_features = ['city', 'education']
num_features = ['age', 'income']

encoder = OrdinalEncoder()
X[cat_features] = encoder.fit_transform(X[cat_features])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42
)

model = xgb.XGBClassifier(
    # Основные параметры
    n_estimators=1000,      # Количество деревьев
    learning_rate=0.1,
    max_depth=6,

    # Регуляризация
    reg_lambda=1.0,         # L2
    reg_alpha=0.0,          # L1
    subsample=0.8,
    colsample_bytree=0.8,

    # Цель и метрики
    objective='binary:logistic',
    eval_metric='auc',

    # Производительность
    tree_method='hist',     # 'hist' или 'gpu_hist'
    random_state=42,
    n_jobs=-1
)

model.fit(X_train,y_train,
          eval_set=[(X_test, y_test)],
          verbose=100,
          early_stopping_rounds=50
)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,

    # Регуляризация
    reg_lambda=3,
    reg_alpha=1,
    min_child_weight=1,
    gamma=0,

    # Стохастика
    subsample=0.8,
    colsample_bytree=0.8,

    # Контроль переобучения
    early_stopping_rounds=50,

    # Метрики и цель
    objective='binary:logistic',
    eval_metric='auc',

    # Производительность
    tree_method='hist',
    n_jobs=-1,
    random_state=42
)

In [ ]:
# GridSearchCV с XGBoost

from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'reg_lambda': [1, 3, 5],
    'n_estimators': [500, 1000]
}

model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")

In [ ]:
# 1. XGBoost НЕ работает напрямую со строковыми категориями
# 2. Используйте OrdinalEncoder или OneHotEncoder
# 3. learning_rate = 0.01–0.05 почти всегда лучше
# 4. Для CPU: tree_method='hist'
# 5. Для GPU: tree_method='gpu_hist'
# 6. Всегда используйте early_stopping_rounds
# 7. Следите за reg_lambda и reg_alpha
# 8. Для дисбаланса: scale_pos_weight

## Optuna with XGBoost 

In [ ]:
import optuna
import xgboost as xgb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder

data = pd.DataFrame({
    'age': [25, 30, 35, 40, 45, 50],
    'city': ['Moscow', 'SPb', 'Moscow', 'Kazan', 'SPb', 'Moscow'],
    'income': [50000, 60000, 55000, 70000, 65000, 80000],
    'education': ['high', 'medium', 'high', 'low', 'medium', 'high'],
    'target': [1, 0, 1, 0, 1, 0]
})

X = data.drop('target', axis=1)
y = data['target']

cat_features = ['city', 'education']
encoder = OrdinalEncoder()
X[cat_features] = encoder.fit_transform(X[cat_features])

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
def objective(trial):

    params = {
        # Основные
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),

        # Регуляризация
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),

        # Стохастика
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),

        # Служебные
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'random_state': 42,
        'n_jobs': -1
    }

    model = xgb.XGBClassifier(**params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        early_stopping_rounds=50,
        verbose=False
    )

    y_pred_proba = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, y_pred_proba)

    return auc


# Запуск оптуна 

study = optuna.create_study(
    direction='maximize',
    study_name='xgboost_auc'
)

study.optimize(objective, n_trials=50)

print('Best AUC:', study.best_value)
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')
    

# Обучение финальной модели

best_model = xgb.XGBClassifier(
    **study.best_params,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)



In [ ]:
# 1. Optuna лучше GridSearch почти всегда
# 2. learning_rate + n_estimators — самая важная связка
# 3. reg_lambda и min_child_weight сильно влияют на переобучение
# 4. Всегда используй early_stopping
# 5. Для GPU: tree_method='gpu_hist'
# 6. Для дисбаланса: scale_pos_weight